In [25]:
import pandas as pd 
import numpy as np
from owlready2 import *

In [26]:
def safe_split(x):
    if type(x) is str:
        return str.split(x,',')
    else:
        return None


aliases = pd.read_csv("../data/utils/aliases.txt", sep="\t")
print(len(aliases))
aliases = aliases[~aliases['Ensembl gene ID'].isna()]
#aliases['Alias symbols'] = aliases['Alias symbols'].apply(lambda x: safe_split(x)) 
#aliases = aliases.explode('Alias symbols')
#print(len(aliases))
aliases.head(1)

50219


,HGNC ID,Approved symbol,Approved name,Status,Previous symbols,Alias symbols,Chromosome,Accession numbers,RefSeq IDs,NCBI Gene ID,Ensembl gene ID
0,HGNC:5,A1BG,alpha-1-B glycoprotein,Approved,NaN,NaN,19q13.43,NaN,NM_130786,1.0,ENSG00000121410


In [27]:
len(aliases) != len(aliases['Ensembl gene ID'].drop_duplicates()) # True -> there are multiple ensembl ids for the 

False

In [28]:
# load open target platform

# read data
files = [f for f in os.listdir("../data/raw/association_by_datasource_direct") if '.parquet' in f]
dfs = []

for f in files:
    dfs.append(pd.read_parquet("../data/raw/association_by_datasource_direct/"+f))

open_targets = pd.concat(dfs, axis = 0)
open_targets.head(2)

,datatypeId,datasourceId,diseaseId,targetId,score,evidenceCount
0,genetic_association,eva,EFO_0000174,ENSG00000135446,0.194538,1
1,genetic_association,eva,EFO_0000174,ENSG00000135452,0.194538,1


In [29]:
open_targets = (
    open_targets
    .merge(
        aliases,
        how='left',
        right_on='Ensembl gene ID',
        left_on='targetId',
        validate='many_to_one'
    )
)

In [30]:
open_targets.head(2)

,datatypeId,datasourceId,diseaseId,targetId,score,evidenceCount,HGNC ID,Approved symbol,Approved name,Status,Previous symbols,Alias symbols,Chromosome,Accession numbers,RefSeq IDs,NCBI Gene ID,Ensembl gene ID
0,genetic_association,eva,EFO_0000174,ENSG00000135446,0.194538,1,HGNC:1773,CDK4,cyclin dependent kinase 4,Approved,NaN,PSK-J3,12q14.1,M14505,NM_000075,1019.0,ENSG00000135446
1,genetic_association,eva,EFO_0000174,ENSG00000135452,0.194538,1,HGNC:10539,TSPAN31,tetraspanin 31,Approved,SAS,NaN,12q14.1,NaN,NaN,6302.0,ENSG00000135452


In [31]:
# load efo ontology
onto_path = "../data/raw/efo.owl"

print("Importing the ontology...")
onto = get_ontology(onto_path).load()
print("Ontology loaded")

Importing the ontology...
Ontology loaded


In [32]:
# all classes in the ontology 
# a class is a node of the hierarchy (concept)
all_classes = [cls for cls in onto.classes()]

dict_classes = {}
for cls in all_classes:
    
    # if gwas_prop[cls]: check if the trait is a gwas_trait
    dict_classes[cls.iri] = {}
    dict_classes[cls.iri]['label'] = cls.label
    dict_classes[cls.iri]['id'] = cls
    dict_classes[cls.iri]['short_id'] = str.split(str(dict_classes[cls.iri]['id']),'.')[-1]
    dict_classes[cls.iri]['description'] = " ".join(cls.IAO_0000115)
    dict_classes[cls.iri]['synonym'] = cls.hasExactSynonym
    #dict_classes[cls.iri]['ancestors'] = [superclasses_dict[a] for a in list(ancestors & set(superclasses_uri))]


In [33]:
def safe_get(x):
    try:
        return x[0]
    except:
        return None

efo_df = pd.DataFrame.from_dict(dict_classes, orient='index')
efo_df['label'] = efo_df['label'].apply(lambda x: safe_get(x))
efo_df['uri'] = efo_df.index
efo_df = efo_df[efo_df['label'] != None]

efo_df.head(1)


,label,id,short_id,description,synonym,uri
http://purl.obolibrary.org/obo/UBERON_0001892,rhombomere,obo.UBERON_0001892,UBERON_0001892,A segment of the developing hindbrain[ZFA]. In...,"[hindbrain neuromere, hindbrain neuromeres, rh...",http://purl.obolibrary.org/obo/UBERON_0001892


In [34]:
superclasses = [
    'http://purl.obolibrary.org/obo/MONDO_0700057', # neurological pain disorder
    'http://purl.obolibrary.org/obo/HP_0012531', # pain
    'http://www.ebi.ac.uk/efo/EFO_0003100', # peripheral neuropathy
]

subclasses_dict = {}

for superclass in superclasses:

    cls = dict_classes[superclass]['id']
    subclasses = list(cls.descendants())

    for s in subclasses:
        subclasses_dict[s.iri] = {}
        subclasses_dict[s.iri]['label'] = s.label
        subclasses_dict[s.iri]['id'] = s
        subclasses_dict[s.iri]['short_id'] = str.split(str(subclasses_dict[s.iri]['id']),'.')[-1]

for k,v in subclasses_dict.items():
    ancestors = list(v['id'].ancestors())
    ancestors = set([a.iri for a in ancestors])
    ancestors = list(ancestors & set(superclasses))
    ancestors = [str(dict_classes[a]['label']) for a in ancestors]
    if(len(ancestors) > 1):
        subclasses_dict[k]['ancestors'] = ",".join(ancestors)
    else:
        subclasses_dict[k]['ancestors'] = ancestors[0]

# subclasses dict is a dictionary with 
# key: iri, value: cls informations of the interesting class 
# all subclasses are relative to defined superclasses

subclasses_df = pd.DataFrame.from_dict(subclasses_dict, orient='index')
subclasses_df.head(2)

,label,id,short_id,ancestors
http://purl.obolibrary.org/obo/MONDO_0021677,[post-infectious neuralgia],obo.MONDO_0021677,MONDO_0021677,"['neurological pain disorder'],['peripheral ne..."
http://www.ebi.ac.uk/efo/EFO_1000854,[causalgia],efo.EFO_1000854,EFO_1000854,['neurological pain disorder']


In [35]:
open_targets_annotated = (
    open_targets
    .merge(
        efo_df.add_suffix('_efo'), 
        left_on="diseaseId",
        right_on="short_id_efo",
        validate='many_to_one',
        how='left'
    )
)
open_targets_annotated.head(2)

,datatypeId,datasourceId,diseaseId,targetId,score,evidenceCount,HGNC ID,Approved symbol,Approved name,Status,...,Accession numbers,RefSeq IDs,NCBI Gene ID,Ensembl gene ID,label_efo,id_efo,short_id_efo,description_efo,synonym_efo,uri_efo
0,genetic_association,eva,EFO_0000174,ENSG00000135446,0.194538,1,HGNC:1773,CDK4,cyclin dependent kinase 4,Approved,...,M14505,NM_000075,1019.0,ENSG00000135446,Ewing sarcoma,efo.EFO_0000174,EFO_0000174,A small round cell tumor that lacks morphologi...,"[ES, Ewing sarcoma, Ewing's family localised t...",http://www.ebi.ac.uk/efo/EFO_0000174
1,genetic_association,eva,EFO_0000174,ENSG00000135452,0.194538,1,HGNC:10539,TSPAN31,tetraspanin 31,Approved,...,NaN,NaN,6302.0,ENSG00000135452,Ewing sarcoma,efo.EFO_0000174,EFO_0000174,A small round cell tumor that lacks morphologi...,"[ES, Ewing sarcoma, Ewing's family localised t...",http://www.ebi.ac.uk/efo/EFO_0000174


In [36]:
open_targets_annotated = (
    open_targets
    .merge(
        efo_df.add_suffix('_efo'), 
        left_on="diseaseId",
        right_on="short_id_efo",
        validate='many_to_one',
        how='left'
    )
)
open_targets_annotated['synonym_efo'] = None
open_targets_annotated['id_efo'] = None
open_targets_annotated.head(2)

,datatypeId,datasourceId,diseaseId,targetId,score,evidenceCount,HGNC ID,Approved symbol,Approved name,Status,...,Accession numbers,RefSeq IDs,NCBI Gene ID,Ensembl gene ID,label_efo,id_efo,short_id_efo,description_efo,synonym_efo,uri_efo
0,genetic_association,eva,EFO_0000174,ENSG00000135446,0.194538,1,HGNC:1773,CDK4,cyclin dependent kinase 4,Approved,...,M14505,NM_000075,1019.0,ENSG00000135446,Ewing sarcoma,None,EFO_0000174,A small round cell tumor that lacks morphologi...,None,http://www.ebi.ac.uk/efo/EFO_0000174
1,genetic_association,eva,EFO_0000174,ENSG00000135452,0.194538,1,HGNC:10539,TSPAN31,tetraspanin 31,Approved,...,NaN,NaN,6302.0,ENSG00000135452,Ewing sarcoma,None,EFO_0000174,A small round cell tumor that lacks morphologi...,None,http://www.ebi.ac.uk/efo/EFO_0000174


In [37]:
open_targets_annotated.to_parquet("../data/quality_control/open_target_annotated.parquet", compression="zstd")

In [38]:
(
    open_targets_annotated[open_targets_annotated['uri_efo'].isin(list(subclasses_dict.keys()))]
    .reset_index(drop=True)
    .to_parquet("../data/quality_control/open_target_annotated_subset.parquet", compression="zstd")
)

In [39]:
open_targets_annotated.columns

Index(['datatypeId', 'datasourceId', 'diseaseId', 'targetId', 'score',
       'evidenceCount', 'HGNC ID', 'Approved symbol', 'Approved name',
       'Status', 'Previous symbols', 'Alias symbols', 'Chromosome',
       'Accession numbers', 'RefSeq IDs', 'NCBI Gene ID', 'Ensembl gene ID',
       'label_efo', 'id_efo', 'short_id_efo', 'description_efo', 'synonym_efo',
       'uri_efo'],
      dtype='str')